In [ ]:
# Phase 0: Data Audit & Schema Verification
Trả lời 10 câu hỏi kiểm toán dữ liệu gốc và xuất bảng Data Dictionary v1.

In [3]:
import warnings
import numpy as np
import pandas as pd
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

RAW_PATH = '../data/raw/steam_games.csv'
df_raw = pd.read_csv(RAW_PATH)

# Câu 1: Số dòng, số cột, kiểm tra trùng lặp mã định danh và tên game
id_col = [
    c
    for c in df_raw.columns
    if 'id' in c.lower() or 'appid' in c.lower()
][0]
name_col = [c for c in df_raw.columns if 'name' in c.lower()][0]

print(
    f'1. Kích thước dữ liệu: {df_raw.shape[0]:,} dòng ×'
    f' {df_raw.shape[1]} cột'
)
print(
    f'   - Số lượng trùng lặp {id_col}:'
    f' {df_raw.duplicated(subset=[id_col]).sum():,}'
)
print(
    f'   - Số lượng trùng lặp {name_col}:'
    f' {df_raw.duplicated(subset=[name_col]).sum():,}'
)

1. Kích thước dữ liệu: 71,429 dòng × 21 cột
   - Số lượng trùng lặp steam_appid: 0
   - Số lượng trùng lặp name: 439


In [4]:
# Câu 2 & Câu 3: Tên chính xác từng cột, kiểu dữ liệu thực tế, % thiếu và mẫu giá trị
audit_df = pd.DataFrame({
    'column_name': df_raw.columns,
    'dtype': df_raw.dtypes.values,
    'null_count': df_raw.isnull().sum().values,
    'null_pct': (df_raw.isnull().sum() / len(df_raw) * 100).round(2).values,
    'empty_list_or_str': [
        df_raw[c].astype(str).isin(['[]', '', 'nan', 'None']).sum()
        for c in df_raw.columns
    ],
    'n_unique': [df_raw[c].nunique(dropna=True) for c in df_raw.columns],
    'sample_values': [
        df_raw[c].dropna().unique()[:3].tolist() for c in df_raw.columns
    ],
})

display(audit_df)

,column_name,dtype,null_count,null_pct,empty_list_or_str,n_unique,sample_values
0,steam_appid,int64,0,0.0,0,71429,"[2719580, 2719590, 2719600]"
1,name,object,0,0.0,0,70990,"[勇者の伝説の勇者, Light No Fire, Lorhaven: Cursed War]"
2,developers,object,0,0.0,200,49545,"[['ぽけそう'], ['Hello Games'], ['GoldenGod Games']]"
3,publishers,object,0,0.0,493,44187,"[['ぽけそう'], ['Hello Games'], ['GoldenGod Games']]"
4,categories,object,0,0.0,558,6995,"[['Single-player', 'Family Sharing'], ['Single-player', 'Multi-player', 'Co-..."
5,genres,object,0,0.0,223,2373,"[['Casual', 'Indie'], ['Action', 'Adventure'], ['RPG', 'Strategy']]"
6,required_age,int64,0,0.0,0,69,"[0, 81, 78]"
7,n_achievements,int64,0,0.0,0,343,"[0, 32, 28]"
8,platforms,object,0,0.0,0,7,"[['windows'], ['windows', 'mac'], ['windows', 'mac', 'linux']]"
9,is_released,bool,0,0.0,0,2,"[True, False]"


In [ ]:
# Câu 4 -> Câu 10: Kiểm tra trường số học, phân phối Mean vs Median (phát hiện ngoại lệ)
num_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
stats_df = df_raw[num_cols].describe(percentiles=[0.25, 0.5, 0.75, 0.99]).T
stats_df['mean_vs_median_ratio'] = (
    stats_df['mean'] / stats_df['50%'].replace(0, np.nan)
).round(2)
stats_df['zero_count'] = (df_raw[num_cols] == 0).sum().values
stats_df['negative_count'] = (df_raw[num_cols] < 0).sum().values

print('--- PHÂN PHỐI CÁC TRƯỜNG SỐ HỌC (KIỂM TRA ĐỘ LỆCH & NGOẠI LỆ) ---')
display(
    stats_df[[
        'count',
        'mean',
        '50%',
        'mean_vs_median_ratio',
        'min',
        '99%',
        'max',
        'zero_count',
        'negative_count',
    ]]
)

--- PHÂN PHỐI CÁC TRƯỜNG SỐ HỌC (KIỂM TRA ĐỘ LỆCH & NGOẠI LỆ) ---


,count,mean,50%,mean_vs_median_ratio,min,99%,max,zero_count,negative_count
steam_appid,71429.0,1.613700e+06,1533470.00,1.05,730.0,3311126.80,3430140.00,0,0
required_age,71429.0,2.573227e+00,0.00,NaN,0.0,80.00,97.00,68906,0
n_achievements,71429.0,1.607948e+01,0.00,NaN,0.0,100.00,9821.00,39896,0
total_reviews,71429.0,4.102497e+02,2.00,205.12,0.0,5129.08,1279700.00,26812,0
total_positive,71429.0,3.602238e+02,2.00,180.11,0.0,4432.12,1107275.00,29562,0
total_negative,71429.0,5.002589e+01,0.00,NaN,0.0,643.72,172425.00,37656,0
review_score,71429.0,2.192849e+00,0.00,NaN,0.0,8.00,9.00,47450,0
positive_percentual,71429.0,4.611329e+01,50.00,0.92,0.0,100.00,100.00,29562,0
metacritic,71429.0,2.573227e+00,0.00,NaN,0.0,80.00,97.00,68906,0
price_initial (USD),71429.0,6.528184e+00,2.99,2.18,0.0,39.99,999.98,22434,0
